### What is ADF

**Azure Data Factory (ADF)** is a cloud-based, serverless **data integration and orchestration service**.

Its Microsoft's cloud-based ETL/ELT and data integration service designed to create data-driven workflows for orchestrating and automating data movement and transformation at scale.

Its primary purpose is to move data from various sources (on-premise databases, SaaS applications, cloud storage) to central repositories (like Azure Data Lake Storage or Synapse) and coordinate workflows.

---
### ADF vs. Azure Databricks

| Feature | Azure Data Factory (ADF) | Azure Databricks |
| --- | --- | --- |
| **Primary Identity** | **Orchestrator & Data Mover** (Workflow engine) | **Data Compute & Processing Engine** (Apache Spark Lakehouse) |
| **Interface / Usability** | Low-code / No-code visual GUI, drag-and-drop pipelines | Code-first collaborative notebooks (PySpark, SQL, Scala, Python) |
| **Data Transformation** | Basic transformations via Copy Activity or Mapping Data Flows (runs behind the scenes on managed Spark clusters, but UI-driven) | Advanced distributed transformations, complex business logic, machine learning, and streaming |
| **Ecosystem Strength** | 100+ native connectors (SAP, Salesforce, Oracle, S3, SQL Server, ADLS) | Deep lakehouse operations: Delta Lake (ACID transactions, Time Travel, Z-Ordering, Auto Loader) |
| **Cost Model** | Pay per activity run, data movement volume, and pipeline execution time | Pay per VM compute uptime (DBUs + Azure Virtual Machine costs) |

---

### Can One Replace the Other?

**In enterprise architectures, generally no—they are complementary, not competing.**

----
#### Why Databricks Cannot Easily Replace ADF:

* **Connectivity & Integration:** ADF connects out-of-the-box to over a hundred enterprise sources (on-premise ERPs, Mainframes via Self-Hosted Integration Runtimes, Salesforce, etc.) with minimal setup. In Databricks, ingesting from legacy or on-prem systems requires custom network routing, JDBC driver maintenance, and boilerplate code.
* **Orchestration Across Multi-Cloud/SaaS:** Databricks Workflows has improved significantly, but ADF is purpose-built to orchestrate non-Spark tasks: triggering Azure Functions, sending Webhooks, running stored procedures, checking file arrivals via event triggers, and managing pipeline failure notifications.

---
#### Why ADF Cannot Easily Replace Databricks:

* **Complex Data Transformations:** Mapping Data Flows in ADF can handle joins, lookups, and basic aggregations, but implementing highly optimized, custom, multi-layered data logic (e.g., healthcare claims grouping, custom deduplication algorithms, ML models) is slow, expensive, and rigid in a drag-and-drop UI compared to native PySpark.
* **Delta Lake Native Capabilities:** ADF lacks deep integration with low-level Delta optimization (like fine-tuned compaction, vacuuming, streaming checkpoints, and custom caching).

---

### How They Work Together (The Standard Pattern)

In modern data engineering (and typical Optum/UHG architectures), the two form a standard pipeline:

1. **ADF** detects a new file drop in ADLS Gen2 or extracts data from an operational source.
2. **ADF** triggers an **Azure Databricks Notebook** activity, passing file paths and dynamic parameters.
3. **Databricks** spins up a Spark cluster, runs the PySpark cleansing, schema validation, and joins, and writes the output into a Delta Lake (Bronze/Silver/Gold).
4. **ADF** receives the execution status back from Databricks and triggers downstream alerting, data quality checks, or BI dataset refreshes.

---
### ADF Key Components

#### 1. Pipelines

- Logical grouping of activities that perform a specific task.
- Can contain data movement, transformation, and control flow activities.
- Support parallel execution, conditional logic, loops, and branching.
- Reusable and parameterizable for flexibility.

#### 2. Activities
- Individual processing steps within a pipeline
- **Types of Activities:**
  - **Data Movement**: Copy Activity (move data between 90+ connectors)
  - **Data Transformation**: Data Flow, Databricks Notebook, HDInsight, Stored Procedure
  - **Control Flow**: If Condition, ForEach, Until, Wait, Execute Pipeline
  - **External**: Web Activity, Webhook, Azure Function

#### 3. Linked Services
- Connection strings that define connection information to data sources
- Similar to connection strings in traditional applications
- Examples: Azure Blob Storage, SQL Database, Databricks, on-premises SQL Server
- Can store credentials in Azure Key Vault for security

#### 4. Datasets
- Named views of data that point to or reference data in linked services
- Represent data structures within data stores (tables, files, folders)
- Define schema, location, and format of data

#### 5. Integration Runtime (IR)

- The compute infrastructure used by ADF to execute activities — it is the *bridge* between ADF and your data sources.
In ADF, the pipeline is just a control plane, while the **Integration Runtime (IR) is the actual execution engine and network bridge**. It does three main things:
   - **Acts as a secure network gateway**—allowing ADF to talk to on-premise systems or private VNets without opening inbound firewall ports via a Self-Hosted IR.
   - **Drives data movement**—reading, serializing, and writing data during Copy activities.
   - **Dispatches and monitors compute**—triggering external workloads like Databricks notebooks or running managed Spark clusters for Mapping Data Flows.

**Why this works:** It demonstrates immediate architecture-level understanding by framing IR around **network security, data transit, and compute orchestration**, rather than just calling it "compute infrastructure.

- **Three Types:**

  ##### Azure IR
  - **What:** Fully managed, serverless compute in Azure cloud
  - **What it's used for:** Cloud-to-cloud data movement, Data Flows (Spark transforms), activity dispatch
  - **When:** All sources/destinations are in the cloud (Azure or public internet)

  ##### Self-hosted IR (SHIR)
  - **What:** Agent you install on your own machine (on-prem server or VM)
  - **What it's used for:** On-prem ↔ cloud data movement, accessing private network/VNet resources
  - **When:** Source data is on-premises or behind a corporate firewall (hybrid scenarios)

  ##### Azure-SSIS IR (SQL Server Integration Services)
  - **What:** Managed Azure VMs hosting the SSIS runtime engine
  - **What it's used for:** Running existing SSIS packages (`.dtsx`) in the cloud with minimal changes
  - **When:** Lift-and-shift migration of legacy SSIS packages without rewriting them

  ##### Quick Comparison

  | Aspect | Azure IR | Self-hosted IR | Azure-SSIS IR |
  | --- | --- | --- | --- |
  | **Hosting** | Fully managed by Azure | Your own machine | Managed Azure VMs |
  | **Best for** | Cloud-to-cloud | On-prem ↔ cloud (hybrid) | Legacy SSIS packages |
  | **Infra mgmt** | None | You manage the host | Minimal |
  | **Network** | Public cloud endpoints | Private network / VNet | Can join VNet |
  | **Cost** | Pay per DIU-hour + volume | Free software (you pay for VM) | Pay per node-hour |

#### 6. Triggers
- Determine when pipeline execution should be initiated
- **Types:**
  - **Schedule Trigger**: Run at specific times (e.g., daily at 2 AM)
  - **Tumbling Window Trigger**: Fixed-size, non-overlapping time intervals with retry/backfill
  - **Event-based Trigger**: Respond to events like blob creation/deletion
  - **Manual Trigger**: On-demand execution

#### 7. Data Flows
- Code-free, visual data transformation interface
- Runs on Spark clusters (abstracted from users)
- Supports 80+ transformations: joins, aggregations, pivots, derived columns, lookups
- Can handle complex ETL logic without writing code

#### 8. Mapping Data Flows vs Wrangling Data Flows
- **Mapping Data Flows**: Scale-out transformations on Spark
- **Wrangling Data Flows**: Power Query integration for self-service data prep

---

### Common Use Cases

1. **Data Migration**
   - Moving data from on-premises to cloud (lift-and-shift)
   - Cloud-to-cloud migrations

2. **Data Lake Ingestion**
   - Loading raw data from various sources into Azure Data Lake
   - Building bronze/silver/gold layer architectures

3. **ETL/ELT Pipelines**
   - Extracting data from multiple sources
   - Transforming data for analytics and reporting
   - Loading into data warehouses (Synapse, SQL DB)

4. **Hybrid Data Integration**
   - Connecting cloud and on-premises data sources
   - Using self-hosted Integration Runtime

5. **Data Orchestration**
   - Coordinating complex workflows across multiple systems
   - Dependency management between tasks
   - Error handling and retry logic

6. **Real-time Data Processing**
   - Event-based triggers for near real-time pipelines
   - Integration with Event Hub, IoT Hub

---

### Integration with Databricks

ADF commonly integrates with Databricks for advanced analytics:

- **Notebook Activity**: Execute Databricks notebooks from ADF pipelines
- **Jar Activity**: Run compiled Spark applications
- **Parameter Passing**: Send runtime parameters from ADF to Databricks
- **Architecture Pattern**: 
  - ADF handles orchestration, scheduling, and data movement
  - Databricks handles complex transformations, ML, and advanced analytics

**Example Workflow:**
```
ADF Pipeline:
1. Copy Activity → Ingest raw data to Data Lake
2. Databricks Notebook Activity → Clean and transform data
3. Copy Activity → Load results to SQL Database
4. Email notification on success/failure
```

---

### Advantages

- **Serverless Architecture**: No infrastructure management required
- **Code-free Interface**: Visual drag-and-drop designer
- **90+ Connectors**: Wide range of source/destination support
- **Both ETL and ELT**: Flexibility in transformation approach
- **Built-in Monitoring**: Pipeline runs, activity logs, alerts
- **CI/CD Support**: Git integration, ARM templates, Azure DevOps
- **Cost-effective**: Pay only for pipeline runs and data movement
- **Security**: VNet integration, private endpoints, managed identity

---

### ADF vs Other Tools

| Feature | ADF | Databricks | SSIS |
| --- | --- | --- | --- |
| Deployment | Cloud-native | Cloud-native | On-premises/Cloud |
| Compute | Serverless | Cluster-based | VM-based |
| Primary Use | Orchestration & ETL | Data engineering & ML | ETL |
| Code Required | Optional | Yes (Python/SQL/Scala) | Visual + C#/VB |
| Cost Model | Pay-per-run | Pay-per-cluster-hour | License + VM cost |

---

### Best Practices

1. **Use Parameters**: Make pipelines reusable and environment-agnostic
2. **Organize with Folders**: Group related pipelines and datasets
3. **Implement Logging**: Use Log Analytics for monitoring and troubleshooting
4. **Error Handling**: Add retry policies and failure notifications
5. **Secure Credentials**: Store secrets in Azure Key Vault
6. **Optimize Copy Activity**: Use parallel copies and compression
7. **Monitor Costs**: Track pipeline runs and data movement volume
8. **Version Control**: Use Git integration for source control
9. **Use Data Flows for Complex Logic**: Instead of multiple Copy activities
10. **Partition Large Datasets**: For better performance

---

### Monitoring and Troubleshooting

- **Monitor Tab**: View pipeline runs, activity runs, trigger runs
- **Alerts**: Set up alerts for pipeline failures
- **Integration with Azure Monitor**: Send logs to Log Analytics workspace
- **Activity Output**: View detailed input/output for each activity
- **Debug Mode**: Test pipelines without triggers

---

### Pricing Considerations

- **Pipeline Orchestration**: Charged per 1,000 activity runs
- **Data Flow Execution**: Charged per vCore-hour
- **Data Movement**: Charged per GB moved (varies by IR type)
- **Pipeline Activities**: Different rates for different activity types
- **Integration Runtime**: Self-hosted IR nodes have hourly charges


### ETL vs ELT: Difference with Example

#### ETL (Extract, Transform, Load):
- Data is extracted from source, transformed in a staging area, then loaded into the target system.
- Transformation happens *before* loading.

#### ELT (Extract, Load, Transform):
- Data is extracted from source, loaded into the target system, then transformed *inside* the target (often a data warehouse or lake).
- Leverages the compute power of modern data platforms.

---

#### Example:

 Step | ETL | ELT |
---|---|---|
 1 | Extract sales data from CRM | Extract sales data from CRM |
 2 | Transform: Clean, aggregate in ETL tool | Load raw data into Azure Synapse |
 3 | Load transformed data into Azure Synapse | Transform: Clean, aggregate using SQL in Synapse |

---

#### When to Use:
- **ETL:** Legacy systems, limited target compute, complex transformations.
- **ELT:** Cloud data warehouses/lakes, scalable compute, large datasets.

---

#### Why ETL shifted to ELT:
- Modern cloud platforms offer powerful, scalable compute in the target system.
- ELT reduces data movement and leverages native processing (e.g., SQL, Spark).
- Simplifies architecture and improves performance for large volumes of data.

### Whats the difference btw `main`(Git branch) and `Live Mode` in ADF?

### 🔀 The Core Difference

| Feature / Aspect | 🌿 Git Mode (`main` / Feature Branch) | ⚡ Live Mode (Data Factory Service) |
| --- | --- | --- |
| **What it represents** | The **source code / draft version** of your pipelines stored in GitHub. | The **running, deployed engine** inside the Azure Data Factory cloud service. |
| **Saving behavior** | Clicking **Save** creates a Git commit in your GitHub repository instantly. | There is no "Save." Changes must pass validation and be **Published** to take effect. |
| **Execution** | Triggers execute from published code, but you can **Debug** test pipelines here safely. | Runs the exact deployed pipelines that scheduled triggers or external calls invoke. |
| **Safety / Drafts** | You can leave pipelines half-finished or with errors without breaking anything. | You **cannot** save broken/invalid code here; ADF forces everything to be valid. |

---

### ⚙️ How They Work Together in Practice

Think of ADF like writing code for a web app:

1. **`main` (or Git Mode) is your Code Editor (VS Code / GitHub):**
* This is where you actually build, edit, and experiment.
* When you add a new activity or dataset, you save it as a JSON file in your GitHub repository.
* If you close your browser, your work is 100% safe in GitHub.


2. **Live Mode is the Production Web Server:**
* This is the actual Azure engine that executes your automated daily schedules (Triggers).
* It does **not** read directly from your `main` branch on every run. It reads from the **published ARM templates** (`adf_publish`).



---

### 🚀 The Workflow: How Changes Move from `main` to Live

When you want your new GitHub changes to actually run on a schedule in Azure:

```
[Edit & Save Code]  ──>  GitHub (`main` branch)  ──>  Click "Publish"  ──>  Live Mode (Deployed Service)

```

1. You build and test your pipeline in **`main`**.
2. When you are happy with it, you click the **Publish** button at the top of the ADF screen.
3. ADF converts all your JSON files from `main` into Azure Resource Manager (ARM) templates, pushes them to `adf_publish`, and updates **Live Mode**.
4. Now, your changes are live in the actual Azure service!

#### 💡 Rule of Thumb:

Always do your daily work, testing, and debugging inside your **Git branch (`main`)**. Only switch to **Live Mode** if you want to inspect what is currently deployed and running live in the cloud environment!

### How do you build an incremental load using the standard Watermark pattern?


#### The 20-Second Analogy (Sets the Hook)

Think of an incremental load like reading a thick novel with a **bookmark**. You don't start from Page 1 every day; you open where the bookmark is, read up to where you stopped today, and move the bookmark forward. In ADF, that bookmark is our **watermark column**."

---

#### The 4-Step Technical Workflow

Deliver the answer in this clear sequence:

1. **Step 1: Check the Old Bookmark (Lookup Activity)**
*"We run a Lookup on our metadata/control table to fetch the last processed cutoff—for example, the timestamp of the last successful run (`OldWatermark`)."*
2. **Step 2: Lock the Current Snapshot (Lookup Activity)**
*"We query the source table to find the current maximum timestamp (`NewWatermark = SELECT MAX(ModifiedDate)`). We lock this upper boundary at the start so we only extract a fixed snapshot of data."*
3. **Step 3: Extract the Delta Slice (Copy Activity)**
"We pull only the slice between the two boundaries:
`WHERE ModifiedDate > OldWatermark AND ModifiedDate <= NewWatermark`.
Then we write this delta batch directly into our target or staging layer."
4. **Step 4: Update the Bookmark (Stored Procedure / Script Activity)**
*"Crucially, chained to the **'On Success'** path of the Copy Activity, we update our control table with the `NewWatermark` value."*

---

#### The Punchline (What Makes You Sound Like a Senior Engineer)

Close your answer with these two production nuances:

* **Why upper bound `<= NewWatermark` matters:**
*"I use an explicit snapshot (`<= NewWatermark`) rather than an open-ended query or `GETDATE()`. If new transactions write to the source while the pipeline is in flight, an open-ended run could silently miss those in-between records and cause data loss."*
* **Fault tolerance:**
*"We only update the control table on successful copy completion. If the pipeline fails midway, the bookmark never moves, making the next rerun safe and repeatable without skipping data."*